# DataSage Stage 1: Data Cleaning GRPO Training

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/training/train_cleaning.ipynb)

Trains **Qwen2.5-3B** via [Unsloth](https://github.com/unslothai/unsloth) + [TRL GRPO](https://huggingface.co/docs/trl/grpo_trainer) to perform multi-domain data cleaning.

The model learns to clean enterprise datasets (HR, Sales, PM, IT Ops) by interacting with the **DataSage Cleaning** OpenEnv environment deployed on HuggingFace Spaces.

**Runtime:** T4 GPU (Colab free tier) | ~30-45 min

---

## 1. Setup

In [ ]:
# Verify GPU
!nvidia-smi
import torch
assert torch.cuda.is_available(), "GPU not detected! Go to Runtime > Change runtime type > T4 GPU"
print(f"GPU: {torch.cuda.get_device_name(0)} ({torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB)")

In [ ]:
%%capture
!pip install unsloth vllm
!pip install trl>=0.26 transformers accelerate peft bitsandbytes
!pip install wandb datasets requests

In [ ]:
# --- API Keys ---
# Option A: Use Colab Secrets (recommended) — add HF_TOKEN and WANDB_API_KEY in the key icon on the left sidebar
# Option B: Paste them below
import os

try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
    print("Loaded keys from Colab Secrets")
except Exception:
    pass  # Fill manually below if not using Colab Secrets

# Uncomment and fill if not using Colab Secrets:
# os.environ["HF_TOKEN"] = "hf_..."
# os.environ["WANDB_API_KEY"] = "wandb_..."

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in Colab Secrets or above"
print("Keys loaded.")

## 2. Load Model (Unsloth 4-bit)

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = "unsloth/Qwen2.5-3B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=2048,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {MODEL_NAME}")
print(f"Parameters: {total:,} total, {trainable:,} trainable ({100*trainable/total:.1f}%)")

## 3. Dataset

64 cleaning prompts across 4 enterprise domains (HR, Sales, Project Management, IT Operations).

In [ ]:
import random
from datasets import Dataset

SYSTEM_PROMPT = (
    "You are a data quality agent. You clean enterprise datasets across multiple "
    "domains (HR, Sales, Project Management, IT Operations).\n\n"
    "Each turn, analyze the data and respond with a JSON cleaning action:\n"
    '{"operation": "<op>", "column": "<col>", "value": "<val>", "params": {}}\n\n'
    "Available operations:\n"
    "- fill_null: Fill null values (value can be \"median\", \"mode\", or a specific value)\n"
    "- fix_type: Fix type mismatches in a column (cast to proper type)\n"
    "- remove_duplicate: Remove duplicate rows\n"
    "- standardize: Standardize column values (strip whitespace, normalize case)\n"
    "- trim: Trim whitespace from column values\n"
    "- correct_typo: Correct typos in categorical values\n\n"
    "Think step by step: examine the data quality report, identify the most "
    "impactful issue, then act."
)

TASK_PROMPTS = [
    # --- HR (16) ---
    "This HR dataset has {n} null values in the MonthlyIncome column. Clean the data.",
    "The HR data has type mismatches in the Age column with values like 'N/A' and '#REF!'. Fix this.",
    "There are duplicate employee records in the HR dataset. Remove them.",
    "The Department column has inconsistent casing: 'sales', 'Sales', 'SALES'. Standardize it.",
    "The JobRole column has leading/trailing whitespace. Trim them.",
    "There are typos in the Attrition column: 'Yse' instead of 'Yes'. Correct them.",
    "The YearsAtCompany column has nulls and string entries like 'unknown'. Fix types then fill nulls.",
    "Multiple DQ issues: nulls in MonthlyIncome, type errors in Age, duplicates. Fix the most impactful first.",
    "The DistanceFromHome column contains '#REF!' and 'TBD' mixed with valid numbers. Clean it.",
    "The OverTime column has '  Yes  ' and 'yes' alongside proper values. Standardize.",
    "The Education column has null values. Fill them with the mode.",
    "The PerformanceRating column has strings like 'High' instead of numeric. Fix types.",
    "The HR dataset has whitespace issues in JobRole. Values like ' Manager ' need trimming.",
    "The Department column has typos: 'Ressearch & Development'. Correct them.",
    "The JobSatisfaction column has nulls and invalid strings. Clean this numeric column.",
    "The HR data quality score is only 0.62. Apply the cleaning operation with the most impact.",
    # --- Sales (16) ---
    "The Sales Amount column has values like 'N/A' and '#REF!'. Fix the type mismatches.",
    "There are {n} null values in the Sales Amount column. Fill them with the median.",
    "The Stage column has inconsistent values: 'prospecting', 'Prospecting', 'PROSPECTING'. Standardize.",
    "Duplicate deal records exist in the sales pipeline. Remove duplicates.",
    "The Region column has whitespace: '  East  ', ' West'. Trim all values.",
    "The Product column has typos: 'GTX Bsaic' instead of 'GTX Basic'. Correct them.",
    "The DaysInStage column has strings like 'unknown' mixed with numbers. Fix types.",
    "The Probability column has nulls and '#REF!' values. Fix this numeric column.",
    "Multiple issues: Amount has nulls, Stage is inconsistent, duplicates exist. Fix highest-impact.",
    "The ForecastCategory column values have extra whitespace. Trim them.",
    "The Sales DQ score is 0.58. The Amount column has the most issues. Clean it.",
    "The Rep column has typos in names. Correct them.",
    "The CloseDate column has invalid formats and nulls. Fix types.",
    "The LeadSource column has inconsistent capitalization. Standardize.",
    "The pipeline has 15% duplicate records skewing forecasts. Remove duplicates.",
    "The Amount column has '-', 'TBD', and 'N/A' alongside valid numbers. Fix type mismatches.",
    # --- Project Management (16) ---
    "The PM dataset has {n} nulls in EstimatedHours. Fill with the median.",
    "The Status column has inconsistent casing: 'in progress', 'In Progress', 'IN PROGRESS'. Standardize.",
    "The ActualHours column has 'N/A' and '#REF!' mixed with numbers. Fix types.",
    "Duplicate task records exist. Remove them.",
    "The Priority column has leading/trailing whitespace. Trim all values.",
    "The RiskFlag column has typos: 'Hgih' instead of 'High'. Correct them.",
    "The CompletionPct column has 'TBD' or 'unknown' values. Fix type mismatches.",
    "Multiple issues: nulls in EstimatedHours, type errors in ActualHours, duplicates. Fix most impactful.",
    "The Assignee column has whitespace issues. Trim and standardize names.",
    "The Milestone column has inconsistent naming. Standardize the values.",
    "The PM DQ score is 0.55. EstimatedHours has the most nulls. Address this.",
    "The Dependencies column has type mismatches and malformed entries. Fix types.",
    "Task records have whitespace in Status causing grouping issues. Trim values.",
    "The ProjectName column has typos. Correct them.",
    "The DueDate column has null values for 20% of tasks. Fill them.",
    "The ActualHours column has both nulls and string values like '-'. Fix types first.",
    # --- IT Operations (16) ---
    "The IT Ops SLATarget column has values like 'N/A'. Fix type mismatches.",
    "There are {n} null values in EscalationLevel. Fill with the median.",
    "The Category column: 'hardware', 'Hardware', 'HARDWARE'. Standardize.",
    "Duplicate tickets exist. Remove duplicates.",
    "The Priority column has whitespace: '  P1-Critical  '. Trim values.",
    "The Status column has typos: 'Resovled' instead of 'Resolved'. Correct them.",
    "The ResolutionType column has inconsistent casing. Standardize.",
    "Multiple issues: SLATarget has type errors, Category is inconsistent, duplicates. Prioritize.",
    "The CustomerImpact column has leading/trailing spaces. Trim all values.",
    "The AffectedSystem column has typos in system names. Correct them.",
    "The IT Ops DQ score is 0.60. Identify and fix the most impactful issue.",
    "The EscalationLevel column has 'unknown' mixed with numbers. Fix types.",
    "The CreatedDate column has null values and invalid formats. Address this.",
    "The Assignee column has whitespace and casing issues. Standardize.",
    "Ticket records have duplicates causing inflated incident counts. Remove duplicates.",
    "The SLATarget column has both nulls and '#REF!' values. Fix types and fill nulls.",
]

random.seed(42)
prompts = [p.format(n=random.randint(5, 45)) for p in TASK_PROMPTS]

dataset = Dataset.from_dict({
    "prompt": [
        [{"role": "system", "content": SYSTEM_PROMPT},
         {"role": "user", "content": p}]
        for p in prompts
    ]
})

print(f"Dataset: {len(dataset)} prompts across 4 domains")

## 4. Reward Functions

Three reward signals:
1. **Environment reward** — steps through the deployed OpenEnv cleaning environment
2. **JSON format reward** — well-formed JSON action output
3. **Reasoning reward** — chain-of-thought before acting

In [ ]:
import json
import re
import requests
import time

ENV_URL = "https://ricalanis-datasage-cleaning.hf.space"

# --- Parser ---
def parse_cleaning_action(text):
    """Extract a cleaning action dict from model output."""
    m = re.search(r'\{[^{}]*"operation"[^{}]*\}', text, re.DOTALL)
    if m:
        try:
            data = json.loads(m.group())
            if "operation" in data and "column" in data:
                return data
        except json.JSONDecodeError:
            pass
    # Fallback: keyword match
    t = text.lower()
    if "fill" in t or "null" in t:
        return {"operation": "fill_null", "column": "", "value": "median", "params": {}}
    if "type" in t or "cast" in t:
        return {"operation": "fix_type", "column": "", "value": "numeric", "params": {}}
    if "duplicate" in t:
        return {"operation": "remove_duplicate", "column": "", "params": {}}
    if "standard" in t:
        return {"operation": "standardize", "column": "", "params": {}}
    if "trim" in t:
        return {"operation": "trim", "column": "", "params": {}}
    if "typo" in t:
        return {"operation": "correct_typo", "column": "", "params": {}}
    return {"operation": "fill_null", "column": "", "value": "median", "params": {}}


# --- Reward 1: Environment ---
_env_calls = 0
_env_total_reward = 0.0

def env_reward_fn(completions, **kwargs):
    """Step through the cleaning environment. Primary reward signal."""
    global _env_calls, _env_total_reward
    rewards = []
    for text in completions:
        try:
            action = parse_cleaning_action(text)
            r = requests.post(f"{ENV_URL}/reset", json={}, timeout=10)
            r = requests.post(f"{ENV_URL}/step", json={"action": action}, timeout=10)
            reward = float(r.json().get("reward", 0.0))
        except Exception as e:
            reward = 0.0
        rewards.append(reward)
    _env_calls += 1
    _env_total_reward += sum(rewards) / len(rewards)
    if _env_calls % 5 == 0:
        print(f"  [env_reward] call {_env_calls}, running avg: {_env_total_reward/_env_calls:.3f}")
    return rewards


# --- Reward 2: JSON format ---
VALID_OPS = {"fill_null", "fix_type", "remove_duplicate", "standardize", "trim", "correct_typo"}

def json_format_reward(completions, **kwargs):
    """Reward well-formed JSON cleaning actions."""
    rewards = []
    for text in completions:
        m = re.search(r'\{[^{}]*"operation"[^{}]*\}', text)
        if m:
            try:
                data = json.loads(m.group())
                if data.get("operation") in VALID_OPS and "column" in data:
                    rewards.append(1.0)
                elif data.get("operation") in VALID_OPS:
                    rewards.append(0.6)
                else:
                    rewards.append(0.3)
            except (json.JSONDecodeError, AttributeError):
                rewards.append(0.2)
        else:
            rewards.append(0.0)
    return rewards


# --- Reward 3: Reasoning ---
def reasoning_reward(completions, **kwargs):
    """Reward chain-of-thought before the action."""
    rewards = []
    for text in completions:
        score = 0.0
        t = text.lower()
        if any(w in t for w in ["first", "let me", "i should", "step 1", "think", "analyze"]):
            score += 0.3
        if any(w in t for w in ["null", "missing", "type", "duplicate", "whitespace", "typo"]):
            score += 0.2
        rewards.append(min(score, 0.5))
    return rewards


# Quick env sanity check
try:
    r = requests.get(f"{ENV_URL}/health", timeout=10)
    print(f"Environment health: {r.status_code}")
except Exception as e:
    print(f"WARNING: Environment not reachable ({e}). Training will use fallback rewards.")

## 5. Train with GRPO

In [ ]:
from trl import GRPOConfig, GRPOTrainer

os.environ["WANDB_PROJECT"] = "datasage"

training_args = GRPOConfig(
    output_dir="./outputs/cleaning-grpo",
    learning_rate=5e-6,
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    num_generations=4,
    max_completion_length=256,
    max_prompt_length=512,
    logging_steps=1,
    save_steps=50,
    bf16=True,
    use_vllm=True,
    vllm_mode="colocate",
    report_to="wandb",
    run_name="datasage-cleaning-grpo",
)

trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset,
    reward_funcs=[
        env_reward_fn,       # Primary: environment reward
        json_format_reward,  # Auxiliary: structured output
        reasoning_reward,    # Auxiliary: chain-of-thought
    ],
)

print(f"Starting GRPO training...")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Generations per prompt: {training_args.num_generations}")
print(f"  Reward functions: env, json_format, reasoning")
print()

t_start = time.time()
trainer.train()
elapsed = (time.time() - t_start) / 60
print(f"\nTraining complete in {elapsed:.1f} minutes")
print(f"Environment reward calls: {_env_calls}, avg reward: {_env_total_reward/_env_calls:.3f}" if _env_calls else "")

## 6. Save & Push to Hub

In [ ]:
HF_MODEL_REPO = "ricalanis/datasage-qwen-cleaning"

trainer.save_model("./outputs/cleaning-grpo-final")
tokenizer.save_pretrained("./outputs/cleaning-grpo-final")
print("Model saved locally.")

trainer.push_to_hub(HF_MODEL_REPO)
print(f"Pushed to https://huggingface.co/{HF_MODEL_REPO}")